# Листок №1. СЛАУ, матрицы, квадратичные формы

Практическое занятие. Ниже — краткая теоретическая справка, а затем отдельные блоки кода для каждой задачи (1.1–1.20). Можно выполнять ячейки по порядку.


In [ ]:
import numpy as np
import scipy.linalg as la
import sympy as sp
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

sp.init_printing()


## Краткая теория

**СЛАУ и rref.** Систему $Ax=b$ удобно решать приведением расширенной матрицы $(A|b)$ к упрощённому ступенчатому виду (RREF — Reduced Row Echelon Form). Столбцы без ведущего элемента соответствуют свободным переменным; через них выражается общее решение.

**Определитель.** $\det A$ — полилинейная кососимметричная функция строк матрицы; вычисляется через разложение по строке/столбцу, приведение к треугольному виду или произведение собственных значений: $\det A=\prod \lambda_i$.

**Обратная матрица.** $A^{-1}$ существует при $\det A\neq0$; для блочных и разреженных матриц выгоднее находить обратную не «в лоб», а пользуясь структурой (блочные формулы, формула Шермана–Моррисона и т.п.), особенно при больших $n$.

**Собственные значения и векторы.** $Av=\lambda v$; характеристический многочлен $\det(A-\lambda I)=0$. Для самосопряжённого (симметричного вещественного) оператора все $\lambda_i\in\mathbb R$ и существует ортонормированный базис из собственных векторов: $A=U\Lambda U^{T}$.

**Минимальный многочлен.** Многочлен наименьшей степени $m(x)$ такой, что $m(A)=0$; делит характеристический многочлен, а для каждого собственного значения его кратность в $m(x)$ равна размеру наибольшей жордановой клетки.

**Квадратичные формы.** $Q(x)=x^{T}Ax$, $A=A^{T}$. Приведение к главным осям — это ортогональная диагонализация $A$. Знакоопределённость проверяется критерием Сильвестра: все угловые миноры положительны (положительная определённость) либо знаки миноров чередуются, начиная с отрицательного (отрицательная определённость). Две формы эквивалентны над $\mathbb R$ (закон инерции Сильвестра), если у них одинаковые ранг и сигнатура (число положительных/отрицательных/нулевых собственных значений).

**Кривые второго порядка.** Общее уравнение $Ax^2+Bxy+Cy^2+Dx+Ey+F=0$ приводится к каноническому виду поворотом (диагонализация матрицы квадратичной части $\begin{pmatrix}A&B/2\\B/2&C\end{pmatrix}$) и параллельным переносом в центр (если он есть).

**Произведение Кронекера и матрицы Адамара.** $H_n H_n^{T}=nI_n$; матрицы Адамара порядка $2^k$ строятся рекурсивно (конструкция Сильвестра) $H_{2k}=H_2\otimes H_{k}$.


### 1.1. Общее решение СЛАУ через rref

In [ ]:
A = sp.Matrix([[1, 5, -2, 0],
               [-3, 1, 9, -5],
               [4, -8, -1, 7]])
b = sp.Matrix([-7, 9, 0])

Ab = A.row_join(b)
rref_matrix, pivots = Ab.rref()

print("RREF(A|b):")
sp.pprint(rref_matrix)
print("Ведущие столбцы:", pivots)


### 1.2. RREF и общее решение (5 переменных)

In [ ]:
A = sp.Matrix([[5, -1, 5, 9, -2],
               [3, 0, 6, 9, -1],
               [2, 0, 4, 6, -1]])
b = sp.Matrix([-3, 4, 0])

Ab = A.row_join(b)
rref_matrix, pivots = Ab.rref()

print("RREF(A|b):")
sp.pprint(rref_matrix)
print("Ведущие столбцы:", pivots)

# Общее решение: свободные переменные - x3, x4
x3, x4 = sp.symbols('x3 x4')
x1 = rref_matrix[0, -1] - rref_matrix[0, 2]*x3 - rref_matrix[0, 3]*x4
x2 = rref_matrix[1, -1] - rref_matrix[1, 2]*x3 - rref_matrix[1, 3]*x4
x5 = rref_matrix[2, -1]
print("x = (", x1, ",", x2, ", x3, x4,", x5, ")")


### 1.3. y как функция от x: y = a + b·x

In [ ]:
x, y, z, u = sp.symbols('x y z u')

eqs = [sp.Eq(32*x + 97*y + 90*z + 48*u, 47),
       sp.Eq(35*x + 84*y + 91*z + 54*u, 40),
       sp.Eq(17*x + 18*y + 21*z + 13*u, 37)]

# Система недоопределена (3 уравнения, 4 неизвестных) -> выражаем y,z,u через x
sol = sp.solve(eqs, [y, z, u], dict=True)[0]

y_expr = sp.expand(sol[y])
print("y(x) =", y_expr)

poly = sp.Poly(y_expr, x)
b, a = poly.all_coeffs()  # y = a + b*x  -> coeffs from highest degree
print("a =", a)
print("b =", b)


### 1.4. Действия над матрицами: M1·M2 + M3

In [ ]:
M1 = sp.Matrix([[3, 0, 2],
                [0, 1, 3],
                [2, 2, 0],
                [0, 1, 0]])

M2 = sp.Matrix([[1, 2, -1, 2],
                [-2, -1, 1, 2],
                [2, 1, 1, 2]])

M3 = sp.Matrix([[0, -4, 6, 1],
                [2, 2, -5, -2],
                [2, -2, 6, 4],
                [1, 3, 0, 1]])

result = M1 * M2 + M3
sp.pprint(result)


### 1.5. Определители спец. вида: min(i,j), max(i,j), |i-j|

In [ ]:
def det_min(n):
    A = sp.Matrix(n, n, lambda i, j: min(i+1, j+1))
    return A.det()

def det_max(n):
    A = sp.Matrix(n, n, lambda i, j: max(i+1, j+1))
    return A.det()

def det_absdiff(n):
    A = sp.Matrix(n, n, lambda i, j: abs(i-j))
    return A.det()

print(f"{'n':>3} {'det(min)':>12} {'det(max)':>12} {'det(|i-j|)':>14}")
for n in range(1, 8):
    print(f"{n:>3} {int(det_min(n)):>12} {int(det_max(n)):>12} {int(det_absdiff(n)):>14}")


### 1.6. Наибольший определитель матрицы 6×6 из {-1,0,1}

Полный перебор здесь неосуществим: вариантов $3^{36}\approx 1.5\cdot10^{17}$. Используем случайный поиск / локальные улучшения, чтобы найти хороший кандидат (известное точное значение для порядка 6 — **160**).

In [ ]:
import random

def random_search(n=6, trials=300000, seed=0):
    rng = np.random.default_rng(seed)
    best = -1
    best_M = None
    for _ in range(trials):
        M = rng.choice([-1, 0, 1], size=(n, n))
        d = abs(round(np.linalg.det(M)))
        if d > best:
            best, best_M = d, M.copy()
    return best, best_M

best, best_M = random_search()
print("Найденный максимум |det| (случайный поиск):", best)
print(best_M)


### 1.7. Матрица 80000×80000: диагональ + два внедиагональных элемента

Матрица $A$ диагональна, кроме элементов $a_{12}=0.57$ и $a_{21}=0.03$. Строить полную матрицу $80000\times80000$ не нужно — определитель и след обратной считаются через блок $2\times2$ (индексы 1,2) и произведение/сумму по остальным диагональным элементам.

In [ ]:
n = 80000
a1, an = 0.99, 1.01
d = (an - a1) / (n - 1)
diag = a1 + d * np.arange(n)   # diag[0]=a11, diag[1]=a22, ..., diag[-1]=ann

a12, a21 = 0.57, 0.03
a11, a22 = diag[0], diag[1]

block = np.array([[a11, a12], [a21, a22]])
det_block = np.linalg.det(block)
# произведение остальных диагональных элементов через сумму логарифмов (устойчивее)
log_prod_rest = np.sum(np.log(diag[2:]))
det_A = det_block * np.exp(log_prod_rest)

inv_block = np.linalg.inv(block)
trace_inv = np.trace(inv_block) + np.sum(1.0 / diag[2:])

print("det(A)      =", det_A)
print("trace(A^-1) =", trace_inv)


### 1.8. Матрица 43000×43000: диагональ (геом. прогрессия) + первая строка (ариф. прогрессия)

Матрица $A$ верхнетреугольная блочного вида $\begin{pmatrix}a_{11}&r^T\\0&D\end{pmatrix}$, где $D=\mathrm{diag}(g_2,\dots,g_n)$. Отсюда явная формула для обратной:
$$B=A^{-1}=\begin{pmatrix}1/a_{11} & -\tfrac{1}{a_{11}}r^TD^{-1}\\0 & D^{-1}\end{pmatrix}.$$

In [ ]:
n = 43000
g1, gn = 1.0, 1.07
ratio = (gn / g1) ** (1.0 / (n - 1))
g = g1 * ratio ** np.arange(n)          # диагональ (геом. прогрессия), g[0]=1, g[-1]=1.07

a11 = g[0]
a1n = 1.10
delta = (a1n - a11) / (n - 1)
row1 = a11 + delta * np.arange(n)       # первая строка (ариф. прогрессия), row1[0]=a11, row1[-1]=1.10

# B[0,0] = 1/a11 ; B[0,j] = -row1[j]/(a11*g[j]) для j>=1 ; B[i,i] = 1/g[i] для i>=1
B_row1 = np.empty(n)
B_row1[0] = 1.0 / a11
B_row1[1:] = -row1[1:] / (a11 * g[1:])

B_diag = 1.0 / g[1:]   # диагональные элементы B при i = 2..n

min_elem = min(B_row1.min(), B_diag.min())
sum_elem = B_row1.sum() + B_diag.sum()

print("Наименьший элемент B :", min_elem)
print("Сумма элементов B    :", sum_elem)


### 1.9. Матричные степенные суммы $S_{m,n}=\sum_{i=m}^{n}A^i$ и $B_m=(I-A)^{-1}A^m$

In [ ]:
gamma = 0.057
A = np.array([[7*gamma, 14*gamma],
              [9*gamma, 1*gamma]])
I = np.eye(2)

def mat_power(A, k):
    vals, vecs = np.linalg.eig(A)
    return (vecs @ np.diag(vals**k) @ np.linalg.inv(vecs)).real

def S(m, n):
    total = np.zeros((2, 2))
    for i in range(m, n+1):
        total += mat_power(A, i)
    return total

S_4_70 = S(4, 70)
print("1) Наибольший элемент S(4,70) =", S_4_70.max())

B4 = np.linalg.inv(I - A) @ mat_power(A, 4)
print("2) Наибольший элемент B4      =", B4.max())

# S(k, n) при n -> inf стремится к A^k (I-A)^-1 (все элементы положительны и растут с n)
k = 1
while True:
    S_k_inf = mat_power(A, k) @ np.linalg.inv(I - A)
    if S_k_inf.max() < 0.0002:
        break
    k += 1
print("3) k =", k)


### 1.10. Определитель как многочлен от x: $ax^2+bx+c$

In [ ]:
x = sp.symbols('x')
M = sp.Matrix([[7, 5, 5, x],
               [2, 2, 6, 9],
               [3, x, 8, 0],
               [3, 1, 1, 0]])

d = sp.expand(M.det())
poly = sp.Poly(d, x)
a, b, c = poly.all_coeffs()

print("det(M) =", d)
print("a =", a)
print("b =", b)
print("c =", c)


### 1.11. Биномиальные коэффициенты по модулю 2 (треугольник Серпинского)

In [ ]:
from math import comb

n = 100
M = np.zeros((n, n), dtype=int)
for i in range(1, n+1):
    for j in range(1, i+1):
        M[i-1, j-1] = comb(i, j) % 2

plt.figure(figsize=(6, 6))
plt.imshow(M, cmap=ListedColormap(['gold', 'royalblue']))
plt.title("Биномиальные коэффициенты mod 2, n=100")
plt.gca().invert_yaxis()
plt.show()


### 1.12. Матрица Адамара: $H_1,H_2,H_4$, произведение Кронекера, $H_{2^6}$

**в)\*** Доказательство $H_{2^k}=H_2\otimes H_{2^{k-1}}$ — по индукции: конструкция Сильвестра задаёт $H_{2m}=\begin{pmatrix}H_m&H_m\\H_m&-H_m\end{pmatrix}=H_2\otimes H_m$ по определению произведения Кронекера; полагая $m=2^{k-1}$, получаем требуемое равенство. База индукции — пункт (б) ниже.

In [ ]:
H1 = np.array([[1]])
H2 = np.array([[1, 1], [1, -1]])
H4 = np.kron(H2, H2)

print("H1 =\n", H1)
print("H2 =\n", H2)
print("H4 =\n", H4)

# проверка ортогональности столбцов: H_n H_n^T = n*I_n
for H in (H1, H2, H4):
    n = H.shape[0]
    print(f"H{n} H{n}^T = {n}*I ? ", np.allclose(H @ H.T, n*np.eye(n)))

# рекурсивная конструкция Сильвестра H_{2^k}
def hadamard_sylvester(k):
    H = np.array([[1]])
    for _ in range(k):
        H = np.block([[H, H], [H, -H]])
    return H

H64 = hadamard_sylvester(6)  # n = 2^6 = 64
print("H4 == H2⊗H2 ?", np.array_equal(H4, hadamard_sylvester(2)))

plt.figure(figsize=(6, 6))
plt.imshow(H64, cmap=ListedColormap(['royalblue', 'orange']))
plt.title("Матрица Адамара H_{2^6} (64x64)")
plt.show()


### 1.13. Собственное значение с наибольшей мнимой частью и собственный вектор

In [ ]:
A = sp.Matrix([[14, 10, 21],
               [12, 15, 26],
               [18, 11, 27]])

eigenvects = A.eigenvects()
Z = max(A.eigenvals().keys(), key=lambda v: sp.im(v))

for val, mult, vecs in eigenvects:
    if sp.simplify(val - Z) == 0:
        v = vecs[0]
        v = v / v[0]        # нормируем так, чтобы первая координата была 1
        X = v
        break

print("Z =", sp.simplify(Z))
print("Im(Z) =", sp.im(Z))
print("X =", X.T)
print("U = Re(X[1]) =", sp.re(X[1]))


### 1.14. Минимальный многочлен матрицы 7×7

In [ ]:
A = sp.Matrix([
    [-3, 7, -6, -1, 8, -3, -2],
    [1, -5, 8, -3, -5, 4, -5],
    [1, 3, -4, 6, -2, 0, 8],
    [-1, 11, -12, 9, 3, -3, 9],
    [-3, 11, -12, 5, 8, -4, 6],
    [-3, 2, 1, -4, 4, 1, -7],
    [1, -5, 6, -2, -4, 3, -2]
])

n = A.shape[0]
x = sp.symbols('x')

charpoly = A.charpoly(x).as_expr()
print("Характеристический многочлен:")
sp.pprint(sp.factor(charpoly))

# Для каждого собственного значения находим наименьшее k, при котором
# ядро (A - lambda*I)^k стабилизируется по размерности -> это кратность в min. многочлене
eigenvals = A.eigenvals()
minpoly = sp.Integer(1)
for lam in eigenvals:
    k = 1
    M = A - lam*sp.eye(n)
    prev_nullity = n - M.rank()
    while True:
        k += 1
        Mk = (A - lam*sp.eye(n))**k
        nullity = n - Mk.rank()
        if nullity == prev_nullity:
            k -= 1
            break
        prev_nullity = nullity
    minpoly *= (x - lam)**k

minpoly = sp.expand(minpoly)
print("Минимальный многочлен:")
sp.pprint(sp.factor(minpoly))

# проверка m(A) = 0: подставляем матрицу A в многочлен m(x)
coeffs = sp.Poly(minpoly, x).all_coeffs()  # от старшей степени к младшей
degree = len(coeffs) - 1
mA = sp.zeros(n)
for power, c in enumerate(reversed(coeffs)):
    mA += c * A**power
print("Проверка m(A) = 0:", mA == sp.zeros(n))


### 1.15. Ортонормированный собственный базис самосопряжённого оператора

In [ ]:
A = sp.Matrix([[17, -8, 4],
               [-8, 17, -4],
               [4, -4, 11]])

basis = []
eigenvalues_diag = []
for val, mult, vecs in A.eigenvects():
    vecs_orth = sp.GramSchmidt(vecs, orthonormal=True)
    for v in vecs_orth:
        basis.append(v)
        eigenvalues_diag.append(val)

P = sp.Matrix.hstack(*basis)
D = sp.diag(*eigenvalues_diag)

print("P (столбцы - собственные векторы) =")
sp.pprint(P)
print("D (матрица в новом базисе) =")
sp.pprint(D)
print("Проверка P^T A P = D:", sp.simplify(P.T * A * P - D) == sp.zeros(3, 3))


### 1.16. Приведение квадратичной формы к главным осям (ортогональное преобразование)

In [ ]:
x1, x2, x3, x4 = sp.symbols('x1 x2 x3 x4')

# f = 3x1^2 + 8x1x2 - 3x2^2 + 4x3^2 - 4x3x4 + x4^2
A = sp.Matrix([[3, 4, 0, 0],
               [4, -3, 0, 0],
               [0, 0, 4, -2],
               [0, 0, -2, 1]])

basis = []
eigenvalues_diag = []
for val, mult, vecs in A.eigenvects():
    vecs_orth = sp.GramSchmidt(vecs, orthonormal=True)
    for v in vecs_orth:
        basis.append(v)
        eigenvalues_diag.append(val)

P = sp.Matrix.hstack(*basis)
D = sp.diag(*eigenvalues_diag)

print("Ортогональная матрица перехода P =")
sp.pprint(P)
print("Канонический вид (собственные значения на диагонали) D =")
sp.pprint(D)
print("Проверка P^T A P = D:", sp.simplify(P.T*A*P - D) == sp.zeros(4, 4))
print("\nКанонический вид формы: f =", " + ".join(f"({lam})*y{i+1}^2" for i, lam in enumerate(eigenvalues_diag)))


### 1.17. Классификация кривых второго порядка

In [ ]:
def classify_conic(A, B, C, D, E, F, label):
    x, y = sp.symbols('x y')
    Qmat = sp.Matrix([[A, sp.Rational(B, 2)], [sp.Rational(B, 2), C]])
    fexpr = A*x**2 + B*x*y + C*y**2 + D*x + E*y + F

    eigenvects = Qmat.eigenvects()
    grad = [sp.diff(fexpr, v) for v in (x, y)]
    center = sp.solve(grad, [x, y], dict=True)

    print(f"--- {label} ---")
    print("Уравнение:", sp.Eq(fexpr, 0))
    print("Матрица квадратичной части:")
    sp.pprint(Qmat)
    print("Собственные значения (коэффициенты канонического вида):")
    for val, mult, vecs in eigenvects:
        print("  lambda =", val, " (кратность", mult, ")")
    if center:
        c = center[0]
        f_center = fexpr.subs(c)
        print("Центр кривой:", c)
        print("Свободный член в центре (F'):", sp.simplify(f_center))
    else:
        print("Центра нет (параболический тип)")
    print()

classify_conic(5, 4, 8, -32, -56, 80, "a) 5x^2+4xy+8y^2-32x-56y+80=0")
classify_conic(5, 12, 0, -22, -12, -19, "b) 5x^2+12xy-22x-12y-19=0")


### 1.18. Значения λ, при которых Q положительно определена

In [ ]:
lam = sp.symbols('lambda', real=True)

# Q = x1^2 + 4x2^2 + x3^2 + 2*lam*x1*x2 + 10*x1*x3 + 6*x2*x3
Q = sp.Matrix([[1, lam, 5],
               [lam, 4, 3],
               [5, 3, 1]])

D1 = Q[0, 0]
D2 = Q[:2, :2].det()
D3 = Q.det()
print("D1 =", D1)
print("D2 =", sp.expand(D2))
print("D3 =", sp.expand(D3))

sol_D2 = sp.solve_univariate_inequality(D2 > 0, lam, relational=False)
sol_D3 = sp.solve_univariate_inequality(D3 > 0, lam, relational=False)
sol = sol_D2.intersect(sol_D3)
print("D1>0 (всегда):", D1 > 0)
print("D2>0 при lambda in", sol_D2)
print("D3>0 при lambda in", sol_D3)
print("Область положительной определённости:", sol)


### 1.19. Значения λ, при которых Q отрицательно определена

In [ ]:
lam = sp.symbols('lambda', real=True)

# Q = lam*x1^2 - 2x2^2 - 3x3^2 + 2x1x2 - 2x1x3 + 2x2x3
Q = sp.Matrix([[lam, 1, -1],
               [1, -2, 1],
               [-1, 1, -3]])

D1 = Q[0, 0]
D2 = sp.expand(Q[:2, :2].det())
D3 = sp.expand(Q.det())
print("D1 =", D1)
print("D2 =", D2)
print("D3 =", D3)

# отрицательная определённость: D1<0, D2>0, D3<0
sol = sp.solve([sp.Lt(D1, 0), sp.Gt(D2, 0), sp.Lt(D3, 0)], lam)
print("Условие отрицательной определённости:", sol)


### 1.20. Эквивалентность квадратичных форм над R (закон инерции Сильвестра)

In [ ]:
# Q1 = 2x1^2+9x2^2+3x3^2+8x1x2-4x1x3-10x2x3
Q1 = sp.Matrix([[2, 4, -2],
                [4, 9, -5],
                [-2, -5, 3]])

# Q2 = 2x1^2+3x2^2+6x3^2-4x1x2-4x1x3+8x2x3
Q2 = sp.Matrix([[2, -2, -2],
                [-2, 3, 4],
                [-2, 4, 6]])

def signature(M):
    eigs = M.eigenvals()
    pos = sum(m for v, m in eigs.items() if v > 0)
    neg = sum(m for v, m in eigs.items() if v < 0)
    zero = sum(m for v, m in eigs.items() if v == 0)
    return pos, neg, zero

sig1 = signature(Q1)
sig2 = signature(Q2)
print("Сигнатура Q1 (+, -, 0):", sig1)
print("Сигнатура Q2 (+, -, 0):", sig2)
print("Эквивалентны над R:", sig1 == sig2)
